In [5]:
import pandas as pd

In [6]:
df=pd.read_csv('telco_churn_encoded.csv')
df.head()

,Gender,Age,Married,Dependents,Number of Dependents,Referred a Friend,Number of Referrals,Tenure in Months,Phone Service,Avg Monthly Long Distance Charges,...,Offer_Offer D,Offer_Offer E,Internet Type_DSL,Internet Type_Fiber Optic,Internet Type_No Internet,Contract_One Year,Contract_Two Year,Payment Method_Credit Card,Payment Method_Mailed Check,Churn
0,1,78,0,0,0,0,0.0,1,0,0.00,...,0,0,1,0,0,0,0,0,0,1
1,0,74,1,1,0,1,1.0,8,1,48.85,...,0,1,0,1,0,0,0,1,0,1
2,1,71,0,1,0,0,0.0,18,1,11.33,...,1,0,0,1,0,0,0,0,0,1
3,0,78,1,1,0,1,1.0,25,1,19.76,...,0,0,0,1,0,0,0,0,0,1
4,0,80,1,1,0,1,1.0,37,1,6.33,...,0,0,0,1,0,0,0,0,0,1


## Baseline - Logistic Regression (Linear)

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

# Features
X = df.drop("Churn", axis=1)

# Target
y = df["Churn"]

# Split into train/test (stratified due to imbalance)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y)

# Initialize Logistic Regression
lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)

# Fit model
lr.fit(X_train, y_train)

# Predictions
y_pred = lr.predict(X_test)
y_probs = lr.predict_proba(X_test)[:, 1]

# Classification report
print(classification_report(y_test, y_pred))

# ROC-AUC
roc = roc_auc_score(y_test, y_probs)
print("ROC-AUC:", roc)

# PR-AUC
pr = average_precision_score(y_test, y_probs)
print("PR-AUC:", pr)

              precision    recall  f1-score   support

           0       0.93      0.79      0.85      1552
           1       0.59      0.83      0.69       561

    accuracy                           0.80      2113
   macro avg       0.76      0.81      0.77      2113
weighted avg       0.84      0.80      0.81      2113

ROC-AUC: 0.893643071099105
PR-AUC: 0.7474140289659099


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


For our main metric, recall, LR performs better out-of-the-box. It captures 83% of actual churners, which is critical for retention strategies.

## Baseline- Random Forest (Tree-Based)

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

# Initialize Random Forest
rf = RandomForestClassifier(
    n_estimators=100,          # default number of trees
    random_state=42,
    class_weight="balanced",   # handles imbalanced classes
    n_jobs=-1                  # use all cores
)

# Fit model
rf.fit(X_train, y_train)

# Predictions
y_pred_rf = rf.predict(X_test)
y_probs_rf = rf.predict_proba(X_test)[:, 1]

# Classification report
print(classification_report(y_test, y_pred_rf))

# ROC-AUC
roc_rf = roc_auc_score(y_test, y_probs_rf)
print("ROC-AUC:", roc_rf)

# PR-AUC
pr_rf = average_precision_score(y_test, y_probs_rf)
print("PR-AUC:", pr_rf)

              precision    recall  f1-score   support

           0       0.86      0.93      0.90      1552
           1       0.75      0.59      0.66       561

    accuracy                           0.84      2113
   macro avg       0.81      0.76      0.78      2113
weighted avg       0.83      0.84      0.83      2113

ROC-AUC: 0.8974211873127882
PR-AUC: 0.7497229309914498


Random Forest improves overall performance and precision, but recall drops to 59%. This demonstrates that more complex models don’t always maximize the metric we care about most.

## Feature Importance

In [9]:
# Feature Importances - RF
importances = rf.feature_importances_
feature_importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print(feature_importance_df.head(10))

                        feature  importance
6           Number of Referrals    0.089874
7              Tenure in Months    0.080822
40            Contract_Two Year    0.078916
22               Monthly Charge    0.071992
23                Total Charges    0.068022
27                Total Revenue    0.065794
26  Total Long Distance Charges    0.054454
12      Avg Monthly GB Download    0.051716
1                           Age    0.051052
28                         CLTV    0.050004


Baseline LR is surprisingly strong for recall, making it a good starting point. RF can still be explored if the goal shifts toward precision or overall predictive power, but for pure recall, LR leads.

Although Random Forest improves overall accuracy and reduces false positives, it misses more churners than Logistic Regression. Since recall is our main metric, LR currently performs better for identifying potential churners. RF may still be valuable if we care about precision or explainability, but for retention-focused recall, LR is the preferred baseline.

#Baseline: Logistic Regression - L1/L2 Regularization

In [10]:
# Logistic Regression with L1
lr_l1 = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    class_weight="balanced",
    max_iter=1000,
    random_state=42)

lr_l1.fit(X_train, y_train)
y_pred_l1 = lr_l1.predict(X_test)
y_probs_l1 = lr_l1.predict_proba(X_test)[:,1]

print("=== Logistic Regression L1 ===\n")
print(classification_report(y_test, y_pred_l1))
print(f"ROC-AUC: {roc_auc_score(y_test, y_probs_l1):.4f}")
print(f"PR-AUC: {average_precision_score(y_test, y_probs_l1):.4f}\n")

# Feature importance (absolute coefficients)
coef_l1 = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": lr_l1.coef_[0]
}).sort_values(by="coefficient", key=abs, ascending=False)
print("Top 5 Features by Absolute Coefficient (L1):")
print(coef_l1.head(5))

# Logistic Regression with L2
lr_l2 = LogisticRegression(
    penalty="l2",
    solver="lbfgs",
    class_weight="balanced",
    max_iter=1000,
    random_state=42)

lr_l2.fit(X_train, y_train)
y_pred_l2 = lr_l2.predict(X_test)
y_probs_l2 = lr_l2.predict_proba(X_test)[:,1]

print("\n=== Logistic Regression L2 ===\n")
print(classification_report(y_test, y_pred_l2))
print(f"ROC-AUC: {roc_auc_score(y_test, y_probs_l2):.4f}")
print(f"PR-AUC: {average_precision_score(y_test, y_probs_l2):.4f}\n")

# Feature importance (absolute coefficients)
coef_l2 = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": lr_l2.coef_[0]
}).sort_values(by="coefficient", key=abs, ascending=False)
print("Top 5 Features by Absolute Coefficient (L2):")
print(coef_l2.head(5))

=== Logistic Regression L1 ===

              precision    recall  f1-score   support

           0       0.94      0.78      0.85      1552
           1       0.59      0.86      0.70       561

    accuracy                           0.80      2113
   macro avg       0.76      0.82      0.78      2113
weighted avg       0.85      0.80      0.81      2113

ROC-AUC: 0.9068
PR-AUC: 0.7786

Top 5 Features by Absolute Coefficient (L1):
              feature  coefficient
40  Contract_Two Year    -2.605770
3          Dependents    -1.494654
39  Contract_One Year    -1.314797
8       Phone Service    -1.301922
5   Referred a Friend     1.251589

=== Logistic Regression L2 ===

              precision    recall  f1-score   support

           0       0.93      0.79      0.85      1552
           1       0.59      0.83      0.69       561

    accuracy                           0.80      2113
   macro avg       0.76      0.81      0.77      2113
weighted avg       0.84      0.80      0.81      

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


The L1 logistic regression captures 86% of churners, outperforming L2. This aligns with our retention-focused objective to identify potential churners. LR L1 as the benchmark baseline to compare.